In [2]:
!pip install langchain>=0.1.17 openai>=1.13.3 langchain_openai>=0.1.6 transformers>=4.40.1 datasets >=2.18.0 accelerate>=0.27.2 sentence-transformers>=2.5.1
!pip install langchain_community
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install llama-cpp-python==0.2.69

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.6
    Uninstalling langchain-text-splitters-0.3.6:
      Successfully uninstalled langchain-text-splitters-0.3.6
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.20
    Uninstalling langchain-0.3.20:
      Successfully uninstalled langchain-0.3.20
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 MB 13.1 MB/s eta 0:00:00
  Installing build depende

In [3]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2025-03-23 14:56:41--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.17, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/41/c8/41c860f65b01de5dc4c68b00d84cead799d3e7c48e38ee749f4c6057776e2e9e/5d99003e395775659b0dde3f941d88ff378b2837a8dc3a2ea94222ab1420fad3?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&Expires=1742745401&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0Mjc0NTQwMX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzQxL2M4LzQxYzg2MGY2NWIwMWRlNWRjNGM2OGIwMGQ4NGNlYWQ3OTlkM2U3YzQ4ZTM4ZWU3NDlmNGM2MDU3Nzc2ZTJlOWUvNWQ5OTAwM2UzOTU3NzU2NTliMGRkZTNmOTQxZ

In [4]:

from langchain import LlamaCpp

llm=LlamaCpp(
    model_path = "Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers =-1,
    max_tokens =500,
    n_ctx =2048,
    seed=42,
    verbose=False
)

In [5]:
llm.invoke("What is 1+2?")

''

# Chain Implementation through Langchain

In [6]:
from langchain import PromptTemplate

template ="""<s><|user|>{input_prompt}<|end|><|assistant|>"""
prompt =PromptTemplate(template=template,input_variables=["input_prompt"])

In [7]:
basic_chain = prompt | llm

In [8]:
basic_chain.invoke(
    {"input_prompt":"HI ! My name is Chaitanya. What is 50 +150?"}
)

' The sum of 50 and 150 is 200. So, 50 + 150 = 200.'

# Multiple Chains


In [9]:
from langchain import LLMChain
template ="""<s><|user|>Create a title for the story about {Summary} Only return the title.<|end|><|assistant|>"""
title_prompt =PromptTemplate(template=template,input_variables=["Summary"])
title = LLMChain(llm=llm,prompt=title_prompt,output_key ="title")

<ipython-input-9-d1265ee61296>:4: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm,prompt=title_prompt,output_key ="title")


In [10]:
title.invoke({"Summary":"A girl that lost her dog"})

{'Summary': 'A girl that lost her dog',
 'title': ' "Lost and Found: The Tale of a Girl\'s Journey to Reunite with Her Beloved Companion"'}

In [11]:
template ="""<s><|user|>Describe the main character for the story about {Summary} with the title {title} Only return two sentences.<|end|><|assistant|>"""
character_prompt=PromptTemplate(template=template,input_variables=["Summary","title"])
character = LLMChain(llm=llm,prompt=character_prompt,output_key ="character")

In [12]:
template ="""<s><|user|>Create a story about {Summary} with the title {title}. The main character is {character}. Only return three sentences.<|end|><|assistant|>"""
story_prompt=PromptTemplate(template=template,input_variables=["Summary","title"])
story = LLMChain(llm=llm,prompt=story_prompt,output_key ="story")

In [13]:
llm_chain = title | character | story

In [14]:
llm_chain.invoke("A girl lost her puppy")

{'Summary': 'A girl lost her puppy',
 'title': ' "Lost and Found: The Tale of a Girl\'s Quest to Recover Her Beloved Puppy"',
 'character': ' The main character in "Lost and Found: The Tale of a Girl\'s Quest to Recover Her Beloved Puppy" is a determined and compassionate young girl, whose unwavering love for her lost puppy drives her on an emotional journey filled with hope, resilience, and self-discovery. As she navigates the challenges of searching for her beloved companion, her character grows stronger in both empathy and problem-solving skills.',
 'story': ' "Lost and Found: The Tale of a Girl\'s Quest to Recover Her Beloved Puppy" follows the heartwarming journey of Emily, a determined young girl who refuses to give up on her beloved puppy, Max. When she realizes that Max has gone missing in their small town, Emily embarks on an emotional and challenging adventure filled with hope, resilience, and self-discovery as she searches every corner of the community for her furry friend. 

# Conversation Buffer

In [25]:
template = """<s><|user|>Current Conversation;{chat_history} {input_prompt} <|end|><|assistant|>"""
prompt =PromptTemplate(template=template,input_variables=["input_prompt","chat_history"])

In [29]:
from langchain.memory import ConversationBufferMemory
from langchain import LLMChain
memory = ConversationBufferMemory(memory_key="chat_history")

llm_chain=LLMChain(prompt=prompt,
                   llm=llm,
                   memory = memory
)


In [30]:
llm_chain.invoke({"input_prompt" : "HI I am Chaitanya. What is Newton's Third Law"})

{'input_prompt': "HI I am Chaitanya. What is Newton's Third Law",
 'chat_history': '',
 'text': " Hi Chaitanya! Newton's Third Law states that for every action, there is an equal and opposite reaction. This means that whenever one object exerts a force on another object, the second object simultaneously exerts a force of equal magnitude in the opposite direction on the first object. For example, if you push against a wall with your hand, the wall pushes back against your hand with an equal force in the opposite direction. This law is fundamental to understanding how forces interact in pairs and plays a crucial role in explaining various phenomena in physics."}

In [31]:
llm_chain.invoke({"input_prompt" :"What is my Name?"})

{'input_prompt': 'What is my Name?',
 'chat_history': "Human: HI I am Chaitanya. What is Newton's Third Law\nAI:  Hi Chaitanya! Newton's Third Law states that for every action, there is an equal and opposite reaction. This means that whenever one object exerts a force on another object, the second object simultaneously exerts a force of equal magnitude in the opposite direction on the first object. For example, if you push against a wall with your hand, the wall pushes back against your hand with an equal force in the opposite direction. This law is fundamental to understanding how forces interact in pairs and plays a crucial role in explaining various phenomena in physics.",
 'text': " Hi there! You're Chaitanya. Newton's Third Law, indeed, describes that for every action (force) in nature, there is an equal and opposite reaction. This law helps us understand the interactions between objects, whether it's how rockets propel forward by expelling gases backward or why you feel pushed wh

#Buffer Window Memory

In [34]:
from langchain.memory import ConversationBufferWindowMemory

memory =ConversationBufferWindowMemory(k=2,memory_key="chat_history")

llm_chain=LLMChain(prompt=prompt,
                   llm=llm,
                   memory = memory
)


In [35]:
llm_chain.invoke({"input_prompt" : "HI I am Chaitanya and I am from Asia. What is Newton's First Law"})
llm_chain.invoke({"input_prompt" :"What is 14 + 16?"})

{'input_prompt': 'What is 14 + 16?',
 'chat_history': "Human: HI I am Chaitanya and I am from Asia. What is Newton's First Law\nAI:  Hello Chaitanya! It's great to meet you. Newton's First Law, also known as the Law of Inertia, states that an object at rest will stay at rest, and an object in motion will continue moving at a constant velocity unless acted upon by an external force. This means that if no net force is applied to an object (if all forces acting on it are balanced), then its state of motion won't change; whether stationary or moving uniformly straight-line motion.",
 'text': " You're welcome! To answer your question, the sum of 14 and 16 is 30. As for Newton's First Law, it indeed plays a crucial role in understanding how objects behave under different circumstances without external influence. Keep exploring physics – it's always fascinating to see how these principles apply universally!\n\nBest regards,\nAI"}

In [36]:
llm_chain.invoke({"input_prompt" :"What is my Name?"})

{'input_prompt': 'What is my Name?',
 'chat_history': "Human: HI I am Chaitanya and I am from Asia. What is Newton's First Law\nAI:  Hello Chaitanya! It's great to meet you. Newton's First Law, also known as the Law of Inertia, states that an object at rest will stay at rest, and an object in motion will continue moving at a constant velocity unless acted upon by an external force. This means that if no net force is applied to an object (if all forces acting on it are balanced), then its state of motion won't change; whether stationary or moving uniformly straight-line motion.\nHuman: What is 14 + 16?\nAI:  You're welcome! To answer your question, the sum of 14 and 16 is 30. As for Newton's First Law, it indeed plays a crucial role in understanding how objects behave under different circumstances without external influence. Keep exploring physics – it's always fascinating to see how these principles apply universally!\n\nBest regards,\nAI",
 'text': " Hi Chaitanya! Your name is mention

In [37]:
llm_chain.invoke({"input_prompt" :"Which continent I'm from?"})

{'input_prompt': "Which continent I'm from?",
 'chat_history': "Human: What is 14 + 16?\nAI:  You're welcome! To answer your question, the sum of 14 and 16 is 30. As for Newton's First Law, it indeed plays a crucial role in understanding how objects behave under different circumstances without external influence. Keep exploring physics – it's always fascinating to see how these principles apply universally!\n\nBest regards,\nAI\nHuman: What is my Name?\nAI:  Hi Chaitanya! Your name is mentioned at the beginning of our conversation. But if you have any specific questions or need further clarification about Newton's First Law or anything else, feel free to ask.\n\nRegarding your math question, 14 + 16 equals 30. If there's anything more on this topic or another subject you're interested in, I'm here to help!\n\nBest regards,\nAI",
 'text': " Hi there! As an AI, I don't have a physical form or origin from any specific continent. However, the technology and programming that powers me are d

# ConversationSummary

In [39]:
summary_prompt_template ="""<s><|user|>Summarize the conversation and update with the new lines
Current Conversation {summary}

New Lines of conversation
 {new_lines}
 New Summary <|end|>
 <|assistant|>"""
summary_prompt=PromptTemplate(template=summary_prompt_template,input_variables=["summary","new_lines"])


In [40]:
from langchain.memory import ConversationSummaryBufferMemory

memory = ConversationSummaryBufferMemory(llm=llm,
                                         memory_key="chat_history",prompts = summary_prompt)

<ipython-input-40-ad2e27d66b9f>:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryBufferMemory(llm=llm,


In [41]:
llm_chain=LLMChain(prompt=prompt,
                   llm=llm,
                   memory = memory
)

In [42]:
llm_chain.invoke({"input_prompt" : "HI I am Chaitanya and I am from Asia. What is Newton's First Law"})
llm_chain.invoke({"input_prompt" :"What is 14 + 16?"})


{'input_prompt': 'What is 14 + 16?',
 'chat_history': "Human: HI I am Chaitanya and I am from Asia. What is Newton's First Law\nAI:  Hello Chaitanya! That's great to know you're interested in physics.\n\nNewton's First Law, also known as the law of inertia, states that an object at rest will remain at rest, and an object in motion will continue moving with a constant velocity unless acted upon by an external force. This means that if there are no net forces acting on an object (i.e., all the forces cancel each other out), it will maintain its current state of motion. In simpler terms, objects don't change their movement or behavior randomly; they only do so when a force causes them to change.\n\nThis law applies equally in your country as well as anywhere else on Earth since it is a fundamental principle governing the natural world around us. I hope this explanation helps you understand Newton's First Law!",
 'text': " Additionally, if an object has no forces acting upon it (like frict

In [43]:
llm_chain.invoke({"input_prompt" :"What is my Name?"})

{'input_prompt': 'What is my Name?',
 'chat_history': "Human: HI I am Chaitanya and I am from Asia. What is Newton's First Law\nAI:  Hello Chaitanya! That's great to know you're interested in physics.\n\nNewton's First Law, also known as the law of inertia, states that an object at rest will remain at rest, and an object in motion will continue moving with a constant velocity unless acted upon by an external force. This means that if there are no net forces acting on an object (i.e., all the forces cancel each other out), it will maintain its current state of motion. In simpler terms, objects don't change their movement or behavior randomly; they only do so when a force causes them to change.\n\nThis law applies equally in your country as well as anywhere else on Earth since it is a fundamental principle governing the natural world around us. I hope this explanation helps you understand Newton's First Law!\nHuman: What is 14 + 16?\nAI:  Additionally, if an object has no forces acting u

In [44]:
llm_chain.invoke({"input_prompt" :"Which continent I'm from?"})

{'input_prompt': "Which continent I'm from?",
 'chat_history': 'Human: HI I am Chaitanya and I am from Asia. What is Newton\'s First Law\nAI:  Hello Chaitanya! That\'s great to know you\'re interested in physics.\n\nNewton\'s First Law, also known as the law of inertia, states that an object at rest will remain at rest, and an object in motion will continue moving with a constant velocity unless acted upon by an external force. This means that if there are no net forces acting on an object (i.e., all the forces cancel each other out), it will maintain its current state of motion. In simpler terms, objects don\'t change their movement or behavior randomly; they only do so when a force causes them to change.\n\nThis law applies equally in your country as well as anywhere else on Earth since it is a fundamental principle governing the natural world around us. I hope this explanation helps you understand Newton\'s First Law!\nHuman: What is 14 + 16?\nAI:  Additionally, if an object has no 